# TP — Sistema Fotovoltaico Off-Grid con Azimut ≠ 0°

Diseño de una instalación solar autónoma (off-grid) para consumo hogareño, con
el panel orientado a un azimut distinto del óptimo. Se calcula el factor de
corrección de irradiancia mediante la geometría solar (no por tabla), a partir
de datos de irradiancia horizontal de **NASA POWER**.

**Estructura:**
1. Datos de entrada del proyecto
2. Consumo del hogar (planilla de cargas)
3. Datos de irradiancia (NASA POWER)
4. Geometría solar y factor de corrección por azimut
5. HSP corregidas mes a mes
6. Dimensionamiento del generador fotovoltaico
7. Dimensionamiento del banco de baterías
8. Selección de controlador de carga e inversor
9. Resumen de resultados


## 1. Datos de entrada del proyecto

Completar con los datos de la consigna del TP.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Ubicación y geometría del arreglo -------------------------------------
UBICACION   = "RP60, Valcheta, Río Negro, Argentina"     # texto libre
LATITUD     = -40.72                  # grados, negativo = hemisferio sur
LONGITUD    = -66.19                  # grados 

INCLINACION = 40.0   # beta [grados], ángulo de tilt del panel respecto a la horizontal
AZIMUT      = 15.0   # gamma [grados], desviación respecto al NORTE geográfico (hemisferio sur)
                      # convención: 0° = mirando al norte (óptimo en HS), negativo = hacia el Este,
                      # positivo = hacia el Oeste. Ajustar signo según lo que pida la cátedra.

ALBEDO = 0.20  # reflectividad del suelo (pasto/tierra ~0.2, hormigón ~0.3, nieve ~0.6-0.8)

print(f"Ubicación: {UBICACION} | Latitud: {LATITUD}° | Azimut panel: {AZIMUT}° | Inclinación: {INCLINACION}°")

Ubicación: RP60, Valcheta, Río Negro, Argentina | Latitud: -40.72° | Azimut panel: 15.0° | Inclinación: 40.0°


## 2. Consumo del hogar (planilla de cargas)

Completar con los artefactos reales del proyecto.

In [3]:
cargas = pd.DataFrame([
    {"Artefacto": "Heladera",              "Potencia_W": 150, "Horas_dia": 8,  "Tipo": "AC"},
    {"Artefacto": "Iluminación LED (x8)",  "Potencia_W": 12,   "Horas_dia": 5,  "Tipo": "AC", "Cantidad": 8},
    {"Artefacto": "TV",                    "Potencia_W": 80,  "Horas_dia": 3,  "Tipo": "AC"},
    {"Artefacto": "Router / módem",        "Potencia_W": 10,  "Horas_dia": 24, "Tipo": "AC"},
    {"Artefacto": "Carga de celulares",    "Potencia_W": 35,  "Horas_dia": 3,  "Tipo": "DC"},
    {"Artefacto": "Bomba de agua",         "Potencia_W": 400, "Horas_dia": 0.5,"Tipo": "AC"},
    {"Artefacto": "Horno microondas",      "Potencia_W": 1200,"Horas_dia": 0.5,"Tipo": "AC"},
])
cargas["Cantidad"] = cargas.get("Cantidad", 1).fillna(1) 
cargas["Wh_dia"] = cargas["Potencia_W"] * cargas["Horas_dia"] * cargas["Cantidad"]

Ed = cargas["Wh_dia"].sum()  # consumo diario total [Wh/día]
Pico_simultaneo = (cargas["Potencia_W"] * cargas["Cantidad"]).sum()  # potencia si todo prendido a la vez [W]

display(cargas)
print(f"\nConsumo diario total Ed = {Ed:.0f} Wh/día  ({Ed/1000:.2f} kWh/día)")
print(f"Potencia simultánea máxima (referencia) = {Pico_simultaneo:.0f} W")

,Artefacto,Potencia_W,Horas_dia,Tipo,Cantidad,Wh_dia
0,Heladera,150,8.0,AC,1.0,1200.0
1,Iluminación LED (x8),12,5.0,AC,8.0,480.0
2,TV,80,3.0,AC,1.0,240.0
3,Router / módem,10,24.0,AC,1.0,240.0
4,Carga de celulares,35,3.0,DC,1.0,105.0
5,Bomba de agua,400,0.5,AC,1.0,200.0
6,Horno microondas,1200,0.5,AC,1.0,600.0



Consumo diario total Ed = 3065 Wh/día  (3.06 kWh/día)
Potencia simultánea máxima (referencia) = 1971 W


## 3. Datos de irradiancia (NASA POWER)

Descargar de **https://power.larc.nasa.gov/data-access-viewer/**:
- Seleccionar "Solar" → parámetro `ALLSKY_SFC_SW_DWN` (irradiancia global horizontal, kWh/m²/día)
- Frecuencia: mensual (climatología, promedio de varios años)
- Punto: `LATITUD`, `LONGITUD` definidas arriba
- Exportar CSV y guardarlo en la misma carpeta que este notebook

Si todavía no tenés el archivo descargado, la celda de abajo usa valores de
ejemplo editables para que el notebook corra igual — **reemplazalos por los
tuyos apenas tengas el CSV**.

In [4]:
import os

CSV_NASA = "nasa_power_data.csv"  # nombre del archivo descargado de NASA POWER

meses = ["ENE","FEB","MAR","ABR","MAY","JUN","JUL","AGO","SEP","OCT","NOV","DIC"]

if os.path.exists(CSV_NASA):
    # El export de NASA POWER trae metadata en las primeras filas: ajustar
    # skiprows según el archivo real (probar 0, 10, 13... hasta que matcheen las columnas)
    df_nasa = pd.read_csv(CSV_NASA, skiprows=10)
    # Adaptar el nombre de columna real del export (suele ser 'ALLSKY_SFC_SW_DWN' o 'ANN'/meses)
    H0_mensual = df_nasa.iloc[0, 1:13].astype(float).values
    print("Datos cargados desde", CSV_NASA)
else:
    # --- VALORES DE EJEMPLO — reemplazar por los del CSV real de NASA POWER ---
    H0_mensual = np.array([6.8, 6.2, 5.0, 3.7, 2.7, 2.2, 2.4, 3.2, 4.3, 5.4, 6.4, 7.1])
    print(f"No se encontró {CSV_NASA}: usando valores de ejemplo (¡reemplazar!)")

irr = pd.DataFrame({"Mes": meses, "H0_kWh_m2_dia": H0_mensual})
display(irr)

Datos cargados desde nasa_power_data.csv


ValueError: All arrays must be of the same length

## 4. Geometría solar y factor de corrección por azimut

Se calcula, para un día representativo de cada mes, el ángulo de incidencia
$\theta$ sobre el plano inclinado y orientado, y el ángulo cenital $\theta_z$
sobre el plano horizontal:

$$\cos\theta = \sin\delta \sin\phi \cos\beta - \sin\delta \cos\phi \sin\beta \cos\gamma
+ \cos\delta \cos\phi \cos\beta \cos\omega
+ \cos\delta \sin\phi \sin\beta \cos\gamma \cos\omega
+ \cos\delta \sin\beta \sin\gamma \sin\omega$$

$$\cos\theta_z = \cos\phi\cos\delta\cos\omega + \sin\phi\sin\delta$$

donde $\phi$ = latitud, $\delta$ = declinación solar, $\beta$ = inclinación,
$\gamma$ = azimut del panel, $\omega$ = ángulo horario.

Con eso se separa la irradiación horizontal medida en **directa + difusa**
(correlación de Erbs, en función del índice de claridad $k_T$) y se
reconstruye la irradiación sobre el plano inclinado con el **modelo
isotrópico de Liu-Jordan (HDKR simplificado)**:

$$H_T = H_b \cdot R_b + H_d \cdot \frac{1+\cos\beta}{2} + H \cdot \rho \cdot \frac{1-\cos\beta}{2}$$

In [ ]:
def dia_representativo(mes_idx):
    '''Día del año representativo de cada mes (método de Klein), mes_idx: 0=ENE ... 11=DIC'''
    dias_rep = [17, 47, 75, 105, 135, 162, 198, 228, 258, 288, 318, 344]
    return dias_rep[mes_idx]

def declinacion(n):
    '''Declinación solar delta [rad], ecuación de Cooper. n: día del año (1-365)'''
    return np.radians(23.45 * np.sin(np.radians(360 * (284 + n) / 365)))

def h0_extraterrestre(n, phi_rad):
    '''Irradiación extraterrestre diaria sobre horizontal [kWh/m2/dia]. n: día del año.'''
    Gsc = 1.367  # kW/m2, constante solar
    delta = declinacion(n)
    ws = np.arccos(-np.tan(phi_rad) * np.tan(delta))  # ángulo horario de salida del sol [rad]
    factor_orb = 1 + 0.033 * np.cos(np.radians(360 * n / 365))
    H0e = (24 / np.pi) * Gsc * factor_orb * (
        np.cos(phi_rad) * np.cos(delta) * np.sin(ws) + ws * np.sin(phi_rad) * np.sin(delta)
    )
    return H0e  # kWh/m2/dia

def factor_Rb(n, phi_rad, beta_rad, gamma_rad):
    \"\"\"Relación entre irradiación directa sobre plano inclinado/horizontal,
    integrada numéricamente sobre las horas de sol del día.\"\"\"
    delta = declinacion(n)
    ws = np.degrees(np.arccos(-np.tan(phi_rad) * np.tan(delta)))
    omegas = np.radians(np.linspace(-ws, ws, 500))  # ángulo horario, paso fino

    cos_tz = np.cos(phi_rad)*np.cos(delta)*np.cos(omegas) + np.sin(phi_rad)*np.sin(delta)
    cos_t  = (np.sin(delta)*np.sin(phi_rad)*np.cos(beta_rad)
              - np.sin(delta)*np.cos(phi_rad)*np.sin(beta_rad)*np.cos(gamma_rad)
              + np.cos(delta)*np.cos(phi_rad)*np.cos(beta_rad)*np.cos(omegas)
              + np.cos(delta)*np.sin(phi_rad)*np.sin(beta_rad)*np.cos(gamma_rad)*np.cos(omegas)
              + np.cos(delta)*np.sin(beta_rad)*np.sin(gamma_rad)*np.sin(omegas))

    cos_tz = np.clip(cos_tz, 0, None)  # sólo horas de sol (cenital positivo)
    cos_t  = np.clip(cos_t, 0, None)   # sólo cuando el panel "ve" al sol

    if cos_tz.sum() == 0:
        return 0.0
    return cos_t.sum() / cos_tz.sum()

phi_rad   = np.radians(LATITUD)
beta_rad  = np.radians(INCLINACION)
gamma_rad = np.radians(AZIMUT)

print("Funciones de geometría solar definidas.")

## 5. HSP corregidas mes a mes

In [ ]:
filas = []
for i, mes in enumerate(meses):
    n = dia_representativo(i)
    H = irr.loc[i, "H0_kWh_m2_dia"]          # irradiación horizontal medida (NASA POWER)
    H0e = h0_extraterrestre(n, phi_rad)       # irradiación extraterrestre horizontal
    kT = np.clip(H / H0e, 0, 1)               # índice de claridad

    # Correlación de Erbs (fracción difusa Hd/H) --------------------------------
    if kT <= 0.22:
        fd = 1.0 - 0.09 * kT
    elif kT <= 0.80:
        fd = 0.9511 - 0.1604*kT + 4.388*kT**2 - 16.638*kT**3 + 12.336*kT**4
    else:
        fd = 0.165
    fd = np.clip(fd, 0, 1)

    Hd = H * fd          # componente difusa
    Hb = H - Hd           # componente directa (haz)

    Rb = factor_Rb(n, phi_rad, beta_rad, gamma_rad)

    # Modelo isotrópico (Liu-Jordan) sobre plano inclinado y orientado
    HT = Hb*Rb + Hd*(1+np.cos(beta_rad))/2 + H*ALBEDO*(1-np.cos(beta_rad))/2

    filas.append({"Mes": mes, "H0_horiz": H, "kT": kT, "Hd": Hd, "Hb": Hb,
                  "Rb": Rb, "HT_inclinado": HT, "HSP": HT})

resultado = pd.DataFrame(filas)
display(resultado.round(3))

mes_critico = resultado.loc[resultado["HSP"].idxmin()]
print(f"\nMes más desfavorable: {mes_critico['Mes']}  →  HSP = {mes_critico['HSP']:.2f} h/día")

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(resultado["Mes"], resultado["H0_horiz"], alpha=0.4, label="Horizontal (NASA POWER)")
ax.bar(resultado["Mes"], resultado["HSP"], alpha=0.8, label=f"Inclinado {INCLINACION}° / Azimut {AZIMUT}°")
ax.set_ylabel("kWh/m²/día (HSP)")
ax.set_title("Irradiación horizontal vs. corregida por tilt+azimut")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Dimensionamiento del generador fotovoltaico

Criterio del **mes más desfavorable**.

In [ ]:
HSP_diseno = resultado["HSP"].min()  # h/día, mes crítico

# Pérdidas del sistema: cableado, suciedad, temperatura, MPPT, inversor, etc.
ETA_SISTEMA = 0.78

Potencia_FV_necesaria = Ed / (HSP_diseno * ETA_SISTEMA)  # Wp

POT_PANEL = 450  # Wp, panel comercial elegido
n_paneles = np.ceil(Potencia_FV_necesaria / POT_PANEL)

print(f"HSP de diseño (mes crítico): {HSP_diseno:.2f} h/día")
print(f"Potencia FV necesaria: {Potencia_FV_necesaria:.0f} Wp")
print(f"Paneles de {POT_PANEL} Wp necesarios: {n_paneles:.0f}  (potencia instalada: {n_paneles*POT_PANEL:.0f} Wp)")

## 7. Dimensionamiento del banco de baterías

In [ ]:
DIAS_AUTONOMIA = 2
V_SISTEMA = 24        # V
DOD = 0.5              # plomo-ácido 0.5 | litio LFP 0.8-0.9
ETA_BATERIA = 0.88     # plomo-ácido ~0.85-0.90 | litio ~0.95-0.98

Ah_necesarios = (Ed * DIAS_AUTONOMIA) / (V_SISTEMA * DOD * ETA_BATERIA)
Wh_banco = Ah_necesarios * V_SISTEMA

print(f"Capacidad necesaria: {Ah_necesarios:.0f} Ah @ {V_SISTEMA} V  ({Wh_banco/1000:.2f} kWh)")

CAP_BATERIA_UNIT_AH = 200  # Ah, batería comercial elegida (a V_SISTEMA o su fracción)
n_baterias_paralelo = np.ceil(Ah_necesarios / CAP_BATERIA_UNIT_AH)
print(f"Baterías de {CAP_BATERIA_UNIT_AH} Ah en paralelo: {n_baterias_paralelo:.0f}")

## 8. Selección de controlador de carga e inversor

In [ ]:
# --- Controlador de carga (MPPT) --------------------------------------------
ISC_PANEL = 12.5      # A, corriente de cortocircuito del panel elegido (datasheet)
STRINGS_PARALELO = 2  # cantidad de strings en paralelo del arreglo FV
MARGEN_SEGURIDAD = 1.25

I_controlador_min = ISC_PANEL * STRINGS_PARALELO * MARGEN_SEGURIDAD
print(f"Corriente mínima del controlador MPPT: {I_controlador_min:.1f} A")

# --- Inversor -----------------------------------------------------------------
FACTOR_ARRANQUE = 3  # motores (heladera/bomba) piden 3-6x la nominal al arrancar

Pot_inversor_nominal = Pico_simultaneo
Pot_inversor_pico = Pico_simultaneo * FACTOR_ARRANQUE

print(f"Inversor — potencia nominal mínima: {Pot_inversor_nominal:.0f} W")
print(f"Inversor — potencia pico/surge mínima: {Pot_inversor_pico:.0f} W")

## 9. Resumen de resultados

In [ ]:
resumen = pd.DataFrame({
    "Parámetro": ["Consumo diario (Ed)", "HSP mes crítico", "Potencia FV instalada",
                  "Banco de baterías", "Controlador MPPT (mín.)", "Inversor nominal (mín.)",
                  "Inversor pico (mín.)"],
    "Valor": [f"{Ed:.0f} Wh/día", f"{HSP_diseno:.2f} h/día", f"{n_paneles*POT_PANEL:.0f} Wp",
              f"{Wh_banco/1000:.2f} kWh ({Ah_necesarios:.0f} Ah @ {V_SISTEMA}V)",
              f"{I_controlador_min:.1f} A", f"{Pot_inversor_nominal:.0f} W",
              f"{Pot_inversor_pico:.0f} W"]
})
display(resumen)